In [1]:
import pandas as pd
import numpy as np

In [2]:
# because training the RF model took more than 1.5 hours, I will only use 50k rows to train the model when performing model selection. 

In [3]:
from pathlib import Path
import os

root = Path.cwd().parent
os.chdir(root)

In [4]:
df = pd.read_csv(r'data\train.csv')

In [5]:
df.drop_duplicates(inplace=True)
df.dropna(inplace=True)

In [6]:
temp_df = df.sample(50000)

In [7]:
ques_df = temp_df[['question1', 'question2']]

In [8]:
ques_df.shape

(50000, 2)

In [9]:
# applying bow

from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer(max_features = 3000)
ques1_arr = vectorizer.fit_transform(ques_df['question1']).toarray()
ques2_arr = vectorizer.fit_transform(ques_df['question2']).toarray()

In [10]:
ques1_arr.shape

(50000, 3000)

In [11]:
# stacking all the words horizontally to build one long array containing words from both questions

X = np.hstack((ques1_arr, ques2_arr))

In [12]:
X.shape

(50000, 6000)

In [13]:
y = temp_df['is_duplicate'].values

### Applying RF, XGB and LogReg 

In [14]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

In [15]:
# train test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42)

In [16]:
rf = RandomForestClassifier()
rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)
print(accuracy_score(y_test, y_pred))

0.7544


In [17]:
xgb = XGBClassifier()
xgb.fit(X_train, y_train)
y_pred = xgb.predict(X_test)
print(accuracy_score(y_test, y_pred))

0.7384


In [18]:
lr = LogisticRegression()
lr.fit(X_train, y_train)
y_pred = lr.predict(X_test)
print(accuracy_score(y_test, y_pred))

c:\Users\apaks\anaconda3\envs\quora-myenv\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


0.7116


### Extra Features

In [19]:
# q1_len - char length
# q2_len - char length
# q1_words - number of words
# q2_words - numbers of words
# words_unique - number of unique words
# words_total - number of total words
# words_common - number of common words

In [14]:
temp_df.head()

,id,qid1,qid2,question1,question2,is_duplicate
42406,42406,76416,76417,What are the strongest majors in terms of job ...,What are the strongest majors in terms of job ...,0
350466,350466,479201,479202,How do I apply online for Allen ASAT exam?,I am Nurture student (Xth Studying). I want to...,0
374273,374273,505161,30183,How do you treat sociopathy and borderline per...,"What is borderline personality disorder, and h...",0
190702,190702,289845,188241,How do I write an essay?,How should one write an essay on myself?,1
187692,187692,286011,286012,How did you get in shape?,What's the fastest way to get in shape for a r...,0


q1_len and q2_len

In [15]:
new_df = temp_df.copy().drop(columns = ['is_duplicate'])

In [16]:
def q1_len(ques):
    return len(ques)

def q2_len(ques):
    return len(ques)

In [17]:
new_df['q1_len'] = new_df['question1'].apply(q1_len)
new_df['q2_len'] = new_df['question2'].apply(q2_len)

In [18]:
new_df.head()

,id,qid1,qid2,question1,question2,q1_len,q2_len
42406,42406,76416,76417,What are the strongest majors in terms of job ...,What are the strongest majors in terms of job ...,107,112
350466,350466,479201,479202,How do I apply online for Allen ASAT exam?,I am Nurture student (Xth Studying). I want to...,42,106
374273,374273,505161,30183,How do you treat sociopathy and borderline per...,"What is borderline personality disorder, and h...",65,67
190702,190702,289845,188241,How do I write an essay?,How should one write an essay on myself?,24,40
187692,187692,286011,286012,How did you get in shape?,What's the fastest way to get in shape for a r...,25,50


q1_words and q2_words

In [19]:
def num_words(ques):
    return len(ques.split())

In [20]:
new_df['q1_words'] = new_df['question1'].apply(num_words)
new_df['q2_words'] = new_df['question2'].apply(num_words)

In [21]:
new_df.head()

,id,qid1,qid2,question1,question2,q1_len,q2_len,q1_words,q2_words
42406,42406,76416,76417,What are the strongest majors in terms of job ...,What are the strongest majors in terms of job ...,107,112,19,20
350466,350466,479201,479202,How do I apply online for Allen ASAT exam?,I am Nurture student (Xth Studying). I want to...,42,106,9,22
374273,374273,505161,30183,How do you treat sociopathy and borderline per...,"What is borderline personality disorder, and h...",65,67,9,11
190702,190702,289845,188241,How do I write an essay?,How should one write an essay on myself?,24,40,6,8
187692,187692,286011,286012,How did you get in shape?,What's the fastest way to get in shape for a r...,25,50,6,11


words_unique

In [22]:
def num_unique_words(text):
    return len(set(text.split()))

In [23]:
ques_combined_text = new_df['question1'] + ' ' + new_df['question2']

In [24]:
new_df['words_unique'] = ques_combined_text.apply(num_unique_words)

words_total

In [25]:
def words_total(text):
    return len(text)

In [26]:
new_df['words_total'] = ques_combined_text.apply(words_total)

In [27]:
new_df.head()

,id,qid1,qid2,question1,question2,q1_len,q2_len,q1_words,q2_words,words_unique,words_total
42406,42406,76416,76417,What are the strongest majors in terms of job ...,What are the strongest majors in terms of job ...,107,112,19,20,17,220
350466,350466,479201,479202,How do I apply online for Allen ASAT exam?,I am Nurture student (Xth Studying). I want to...,42,106,9,22,23,149
374273,374273,505161,30183,How do you treat sociopathy and borderline per...,"What is borderline personality disorder, and h...",65,67,9,11,17,133
190702,190702,289845,188241,How do I write an essay?,How should one write an essay on myself?,24,40,6,8,11,65
187692,187692,286011,286012,How did you get in shape?,What's the fastest way to get in shape for a r...,25,50,6,11,15,76


words_common

In [28]:
def words_common(ques1, ques2):
    return len(set(ques1.split()) & set(ques2.split()))

In [29]:
new_df['words_common'] = new_df.apply(lambda row: words_common(row['question1'], row['question2']), axis = 1)

In [30]:
ques1 = 'How Are you?'
ques2 = 'Are you okay?'
words_common(ques1, ques2)

1

### Applying algorithms again on new_df

In [37]:
new_df.shape

(50000, 12)

In [38]:
new_df.iloc[:, 5:].values

array([[ 58,  54,  10, ...,  11, 113,   9],
       [ 78,  91,  11, ...,  13, 170,  10],
       [ 35,  31,   7, ...,  10,  67,   4],
       ...,
       [ 81,  93,  15, ...,  24, 175,   7],
       [ 43,  71,   8, ...,  15, 115,   5],
       [ 88, 245,  18, ...,  54, 334,   7]], shape=(50000, 7))

In [39]:
X = np.hstack((X, new_df.iloc[:, 5:].values))

In [40]:
X.shape

(50000, 6007)

In [41]:
# train test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42)

In [42]:
rf = RandomForestClassifier()
rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)
print(accuracy_score(y_test, y_pred))

0.7676


In [43]:
xgb = XGBClassifier()
xgb.fit(X_train, y_train)
y_pred = xgb.predict(X_test)
print(accuracy_score(y_test, y_pred))

0.7678


- accuracy improved after adding extra features
- with bow only --> RF: 0.75, XGB:0.73
- with bow and extra features --> RF:0.7676, XGB:0.7678

### Text preprocessing

In [31]:
new_df.shape

(50000, 12)

In [32]:
import re
from bs4 import BeautifulSoup

def preprocess(ques):
    
    # lower case
    ques = str(ques).lower().strip()

    # replace special characters with their names
    ques = ques.replace('%', 'percent')
    ques = ques.replace('$', ' dollar ')
    ques = ques.replace('₹', ' rupee ')
    ques = ques.replace('€', ' euro ')
    ques = ques.replace('@', ' at ')

    # word math appears a lot in the data
    ques = ques.replace('[math]', '')
    
    # convert large numbers intos words
    ques = ques.replace(',000,000,000 ', 'b ')
    ques = ques.replace(',000,000 ', 'm ')
    ques = ques.replace(',000 ', 'k ')
    ques = re.sub(r'([0-9]+)000000000', r'\1b', ques)
    ques = re.sub(r'([0-9]+)000000', r'\1m', ques)
    ques = re.sub(r'([0-9]+)000', r'\1k', ques)

    # expand contractions
    #https://stackoverflow.com/questions/19790188/expanding-english-language-contractions-in-python/19794953#19794953

    contractions = { 
    "ain't": "am not",
    "aren't": "are not",
    "can't": "can not",
    "can't've": "can not have",
    "'cause": "because",
    "could've": "could have",
    "couldn't": "could not",
    "couldn't've": "could not have",
    "didn't": "did not",
    "doesn't": "does not",
    "don't": "do not",
    "hadn't": "had not",
    "hadn't've": "had not have",
    "hasn't": "has not",
    "haven't": "have not",
    "he'd": "he would",
    "he'd've": "he would have",
    "he'll": "he will",
    "he'll've": "he will have",
    "he's": "he is",
    "how'd": "how did",
    "how'd'y": "how do you",
    "how'll": "how will",
    "how's": "how is",
    "i'd": "i would",
    "i'd've": "i would have",
    "i'll": "i will",
    "i'll've": "i will have",
    "i'm": "i am",
    "i've": "i have",
    "isn't": "is not",
    "it'd": "it would",
    "it'd've": "it would have",
    "it'll": "it will",
    "it'll've": "it will have",
    "it's": "it is",
    "let's": "let us",
    "ma'am": "madam",
    "mayn't": "may not",
    "might've": "might have",
    "mightn't": "might not",
    "mightn't've": "might not have",
    "must've": "must have",
    "mustn't": "must not",
    "mustn't've": "must not have",
    "needn't": "need not",
    "needn't've": "need not have",
    "o'clock": "of the clock",
    "oughtn't": "ought not",
    "oughtn't've": "ought not have",
    "shan't": "shall not",
    "sha'n't": "shall not",
    "shan't've": "shall not have",
    "she'd": "she would",
    "she'd've": "she would have",
    "she'll": "she will",
    "she'll've": "she will have",
    "she's": "she is",
    "should've": "should have",
    "shouldn't": "should not",
    "shouldn't've": "should not have",
    "so've": "so have",
    "so's": "so as",
    "that'd": "that would",
    "that'd've": "that would have",
    "that's": "that is",
    "there'd": "there would",
    "there'd've": "there would have",
    "there's": "there is",
    "they'd": "they would",
    "they'd've": "they would have",
    "they'll": "they will",
    "they'll've": "they will have",
    "they're": "they are",
    "they've": "they have",
    "to've": "to have",
    "wasn't": "was not",
    "we'd": "we would",
    "we'd've": "we would have",
    "we'll": "we will",
    "we'll've": "we will have",
    "we're": "we are",
    "we've": "we have",
    "weren't": "were not",
    "what'll": "what will",
    "what'll've": "what will have",
    "what're": "what are",
    "what's": "what is",
    "what've": "what have",
    "when's": "when is",
    "when've": "when have",
    "where'd": "where did",
    "where's": "where is",
    "where've": "where have",
    "who'll": "who will",
    "who'll've": "who will have",
    "who's": "who is",
    "who've": "who have",
    "why's": "why is",
    "why've": "why have",
    "will've": "will have",
    "won't": "will not",
    "won't've": "will not have",
    "would've": "would have",
    "wouldn't": "would not",
    "wouldn't've": "would not have",
    "y'all": "you all",
    "y'all'd": "you all would",
    "y'all'd've": "you all would have",
    "y'all're": "you all are",
    "y'all've": "you all have",
    "you'd": "you would",
    "you'd've": "you would have",
    "you'll": "you will",
    "you'll've": "you will have",
    "you're": "you are",
    "you've": "you have"
    }

    ques_decontracted = []

    for word in ques.split():
        if word in contractions:
            word = contractions[word]
        
        ques_decontracted.append(word)

    ques = ' '.join(ques_decontracted)
    
    ques = ques.replace("'ve", " have")
    ques = ques.replace("n't", " not")
    ques = ques.replace("'re", " are")
    ques = ques.replace("'ll", " will")
    ques = ques.replace("n't", " not")

    # remove html characters 
    ques = BeautifulSoup(ques)
    ques = ques.get_text()

    # remove punctuation marks
    pattern = re.compile(r'\W')
    ques = re.sub(pattern, ' ', ques).strip()

    return ques

In [33]:
new_df['question1'] = new_df['question1'].apply(preprocess)
new_df['question2'] = new_df['question2'].apply(preprocess)

In [34]:
new_df.head()

,id,qid1,qid2,question1,question2,q1_len,q2_len,q1_words,q2_words,words_unique,words_total,words_common
42406,42406,76416,76417,what are the strongest majors in terms of job ...,what are the strongest majors in terms of job ...,107,112,19,20,17,220,16
350466,350466,479201,479202,how do i apply online for allen asat exam,i am nurture student xth studying i want to...,42,106,9,22,23,149,2
374273,374273,505161,30183,how do you treat sociopathy and borderline per...,what is borderline personality disorder and h...,65,67,9,11,17,133,3
190702,190702,289845,188241,how do i write an essay,how should one write an essay on myself,24,40,6,8,11,65,3
187692,187692,286011,286012,how did you get in shape,what is the fastest way to get in shape for a ...,25,50,6,11,15,76,2


### Advanced Features

Token Features
- cwc_min -> ratio of number of common words to the length of smaller question
- cwc_max -> ration of num of common words to the length of bigger question
- csc_min -> ratio of common stop words to the min(stop words in q1 and q2)
- csc_max -> ratio of common stop words to the max(stop words in q1 and q2)
- ctc_min -> ratio of common tokens to the min(number of tokens in q1 and q2)
- ctc_max -> ratio of common tokens to the max(number of tokens in q1 and q2)
- last_word_eq -> boolean: if last word is same
- first_word_eq -> boolean: if first word is same

Length based features
- mean_len -> mean length of the 2 questions
- abs_length_diff -> difference between the absolute length 
- longest_sbstring_ratio -> length of the longest substring/length of the smaller question

FuzzyWuzzy features
- fuzz_ratio -> String Similarity
- fuzz_partial_ratio -> Partial String Similarity
- fuzz_token_sort_ratio -> token_sort_ratio from fuzzywuzzy
- fuzz_set_ratio -> token_set_ratio from fuzzywuzzy

In [ ]:
#token based features

import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords

def fetch_token_features(row):

    q1 = row['question1']
    q2 = row['question2']

    STOP_WORDS = set(stopwords.words('english'))

    # define a list of features with 8 zeros prepopulated
    token_features = [0.0]*8

    # convert the sentence into tokens 
    q1_tokens = q1.split()
    q2_tokens = q2.split()

    if len(q1_tokens) == 0 or len(q2_tokens) == 0:
        return token_features

    # non-stop words in questions
    q1_words = set([word for word in q1_tokens if word not in STOP_WORDS])
    q2_words = set([word for word in q2_tokens if word not in STOP_WORDS])

    # stop words in questions
    q1_stops = set([word for word in q1_tokens if word in STOP_WORDS])
    q2_stops = set([word for word in q2_tokens if word in STOP_WORDS])

    # common words count
    common_words_count = len(q1_words.intersection(q2_words))

    # common stop words count
    common_stop_count = len(q1_stops.intersection(q2_stops))

    # common token counts
    common_token_count = len(set(q1_tokens).intersection(set(q2_tokens)))

    SAFE_DIV = 0.0001
    token_features[0] = common_words_count / (min(len(q1_words), len(q2_words)) + SAFE_DIV)     #ratio of number of common words to the length of smaller question
    token_features[1] = common_words_count / (max(len(q1_words), len(q2_words)) + SAFE_DIV)     #ratio of num of common words to the length of bigger question
    token_features[2] = common_stop_count / (min(len(q1_stops), len(q2_stops)) + SAFE_DIV)     #ratio of common stop words to the min(stop words in q1 and q2)
    token_features[3] = common_stop_count / (max(len(q1_stops), len(q2_stops)) + SAFE_DIV)      #ratio of common stop words to the max(stop words in q1 and q2)
    token_features[4] = common_token_count / (min(len(q1_tokens), len(q2_tokens)) + SAFE_DIV)       # ratio of common tokens to the min(number of tokens in q1 and q2)
    token_features[5] = common_token_count / (max(len(q1_tokens), len(q2_tokens)) + SAFE_DIV)       #ratio of common tokens to the max(number of tokens in q1 and q2)
    token_features[6] = int(q1_tokens[-1] == q2_tokens[-1])     #boolean: if last word is same
    token_features[7] = int(q1_tokens[0] == q2_tokens[0])#boolean: if first word is same

    return token_features



[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\apaks\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [36]:
token_features = new_df.apply(fetch_token_features, axis=1)

In [55]:
token_features_df = pd.DataFrame(token_features.tolist(), columns=['cwc_min', 'cwc_max', 'csc_min', 'csc_max', 'ctc_min', 'ctc_max', 'last_word_eq', 'first_word_eq'], index = new_df.index)

In [56]:
token_features_df.head()

,cwc_min,cwc_max,csc_min,csc_max,ctc_min,ctc_max,last_word_eq,first_word_eq
42406,0.999988,0.888879,0.999986,0.999986,0.789470,0.749996,1.0,1.0
350466,0.399992,0.199998,0.249994,0.166664,0.333330,0.136363,0.0,0.0
374273,0.499988,0.399992,0.499988,0.285710,0.444440,0.363633,0.0,0.0
190702,0.999950,0.666644,0.499988,0.399992,0.666656,0.499994,0.0,1.0
187692,0.999950,0.399992,0.249994,0.142855,0.499992,0.249998,0.0,0.0


In [57]:
new_df.head()

,id,qid1,qid2,question1,question2,q1_len,q2_len,q1_words,q2_words,words_unique,words_total,words_common
42406,42406,76416,76417,what are the strongest majors in terms of job ...,what are the strongest majors in terms of job ...,107,112,19,20,17,220,16
350466,350466,479201,479202,how do i apply online for allen asat exam,i am nurture student xth studying i want to...,42,106,9,22,23,149,2
374273,374273,505161,30183,how do you treat sociopathy and borderline per...,what is borderline personality disorder and h...,65,67,9,11,17,133,3
190702,190702,289845,188241,how do i write an essay,how should one write an essay on myself,24,40,6,8,11,65,3
187692,187692,286011,286012,how did you get in shape,what is the fastest way to get in shape for a ...,25,50,6,11,15,76,2


In [60]:
new_df = pd.concat([new_df, token_features_df], axis=1)

In [61]:
new_df.head()

,id,qid1,qid2,question1,question2,q1_len,q2_len,q1_words,q2_words,words_unique,words_total,words_common,cwc_min,cwc_max,csc_min,csc_max,ctc_min,ctc_max,last_word_eq,first_word_eq
42406,42406,76416,76417,what are the strongest majors in terms of job ...,what are the strongest majors in terms of job ...,107,112,19,20,17,220,16,0.999988,0.888879,0.999986,0.999986,0.789470,0.749996,1.0,1.0
350466,350466,479201,479202,how do i apply online for allen asat exam,i am nurture student xth studying i want to...,42,106,9,22,23,149,2,0.399992,0.199998,0.249994,0.166664,0.333330,0.136363,0.0,0.0
374273,374273,505161,30183,how do you treat sociopathy and borderline per...,what is borderline personality disorder and h...,65,67,9,11,17,133,3,0.499988,0.399992,0.499988,0.285710,0.444440,0.363633,0.0,0.0
190702,190702,289845,188241,how do i write an essay,how should one write an essay on myself,24,40,6,8,11,65,3,0.999950,0.666644,0.499988,0.399992,0.666656,0.499994,0.0,1.0
187692,187692,286011,286012,how did you get in shape,what is the fastest way to get in shape for a ...,25,50,6,11,15,76,2,0.999950,0.399992,0.249994,0.142855,0.499992,0.249998,0.0,0.0


In [ ]:
# length based features

import distance

def fetch_length_features(row):

    q1 = row['question1']
    q2 = row['question2']

    length_features = [0.0]*3

    q1_tokens = q1.split()
    q2_tokens = q2.split()

    if len(q1_tokens) == 0 or len(q2_tokens) == 0:
        return length_features
    
    # absolute length diff
    length_features[0] = abs(len(q1_tokens) - len(q2_tokens))

    # mean length
    length_features[1] = (len(q1_tokens) + len(q2_tokens))/2

    #Avereage token length of both questions
    common_substring = list(distance.lcsubstrings(q1, q2))
    if common_substring:
        length_features[2] = len(common_substring[0]) / (min(len(q1), len(q2)) + 1)
    else:
        length_features[2] = 0.0

    return length_features
    

In [79]:
new_df.iloc[0]

id                                                           42406
qid1                                                         76416
qid2                                                         76417
question1        what are the strongest majors in terms of job ...
question2        what are the strongest majors in terms of job ...
q1_len                                                         107
q2_len                                                         112
q1_words                                                        19
q2_words                                                        20
words_unique                                                    17
words_total                                                    220
words_common                                                    16
cwc_min                                                   0.999988
cwc_max                                                   0.888879
csc_min                                                   0.99

In [80]:
fetch_length_features(new_df.iloc[0])

[1, 19.5, 0.8504672897196262]

In [81]:
new_df.head()

,id,qid1,qid2,question1,question2,q1_len,q2_len,q1_words,q2_words,words_unique,words_total,words_common,cwc_min,cwc_max,csc_min,csc_max,ctc_min,ctc_max,last_word_eq,first_word_eq
42406,42406,76416,76417,what are the strongest majors in terms of job ...,what are the strongest majors in terms of job ...,107,112,19,20,17,220,16,0.999988,0.888879,0.999986,0.999986,0.789470,0.749996,1.0,1.0
350466,350466,479201,479202,how do i apply online for allen asat exam,i am nurture student xth studying i want to...,42,106,9,22,23,149,2,0.399992,0.199998,0.249994,0.166664,0.333330,0.136363,0.0,0.0
374273,374273,505161,30183,how do you treat sociopathy and borderline per...,what is borderline personality disorder and h...,65,67,9,11,17,133,3,0.499988,0.399992,0.499988,0.285710,0.444440,0.363633,0.0,0.0
190702,190702,289845,188241,how do i write an essay,how should one write an essay on myself,24,40,6,8,11,65,3,0.999950,0.666644,0.499988,0.399992,0.666656,0.499994,0.0,1.0
187692,187692,286011,286012,how did you get in shape,what is the fastest way to get in shape for a ...,25,50,6,11,15,76,2,0.999950,0.399992,0.249994,0.142855,0.499992,0.249998,0.0,0.0


In [83]:
length_features = new_df.apply(fetch_length_features, axis = 1)

In [88]:
length_features_df = pd.DataFrame(length_features.tolist(), columns = ['abs_len_diff', 'mean_len', 'longest_substr_ratio'], index = new_df.index)

In [89]:
length_features_df

,abs_len_diff,mean_len,longest_substr_ratio
42406,1.0,19.5,0.850467
350466,13.0,15.5,0.285714
374273,2.0,10.0,0.492308
190702,2.0,7.0,0.625000
187692,6.0,9.0,0.520000
...,...,...,...
97177,1.0,8.5,0.567568
302174,1.0,9.5,0.682927
321200,4.0,10.0,0.510204
66594,0.0,9.0,0.439024


In [91]:
new_df = pd.concat([new_df, length_features_df], axis = 1)

In [98]:
# fuzzywuzzy features

from fuzzywuzzy import fuzz

def fetch_fuzzy_features(row):

    q1 = row['question1']
    q2 = row['question2']

    fuzzy_features = [0.0]*4

    # fuzzy ratio 
    fuzzy_features[0] = fuzz.ratio(q1, q2)

    # fuzz_partial_ratio
    fuzzy_features[1] = fuzz.partial_ratio(q1, q2)

    # token sort ration
    fuzzy_features[2] = fuzz.token_sort_ratio(q1, q2)

    # token set ratio
    fuzzy_features[3] = fuzz.token_set_ratio(q1, q2)

    return fuzzy_features

In [99]:
fuzzy_features = new_df.apply(fetch_fuzzy_features, axis = 1)

In [100]:
fuzzy_features = fuzzy_features.tolist()

In [103]:
fuzzy_features_df = pd.DataFrame(fuzzy_features, columns = ['fuzz_ratio', 'fuzz_partial_ratio', 'token_sort_ratio', 'token_set_ratio'], index = new_df.index)

In [105]:
new_df = pd.concat([new_df, fuzzy_features_df], axis = 1)

In [109]:
new_df['is_duplicate'] = temp_df['is_duplicate']

In [110]:
new_df.to_csv('data/post_feature_engineering_data.csv', index = False)